In [1]:
"""
Step 5c: does the crosstalk benefit need the WMF/RMF front end?

Same synthetic 5-qubit multiplexed bus as step5b (same crosstalk model, no wrap-around, disjoint pools, paired draws).
Two feature families per qubit:
  BOX : tuned-window boxcar after demodulation   -> 2 features per qubit [sum I, sum Q]   (window BOX_WIN)
  DF  : dual filter (WMF + RMF), window [0, 500)  -> 4 features per qubit
Six architectures:
  bx_single   : own boxcar features only
  bx_lin10    : boxcar features of all 5 qubits (cancels LINEAR crosstalk)
  bx_cross    : bx_lin10 + own self-quadratic terms + cross-products with the 4 other qubits
  df_single   : own dual-filter features only
  df_lin20    : dual-filter features of all 5 qubits
  df_cross    : df_lin20 + 10 self-quad terms + 16 cross-products   (your step5 design)

SYNTHETIC data: results tell you how the machinery behaves, not what a real chip does.
"""
import os
import numpy as np

# ------------------------------------------------------------------ config
DATA_DIR = r"D:\MSU\Qubit_readout_FPGA\python"
N_QUBITS = 5
FS = 500.0e6
IF_FREQS = np.array([64.729e6, 25.366e6, -24.790e6, -70.269e6, -127.282e6])
T_BINS = 500
BOX_WIN = (50, 500)                  # boxcar window in bins (tuned on single-qubit data: start ~50, drop the ring-up)
EMA_ALPHA = 0.05
RIDGE_LAMBDA_REL = 1e-4
CLIP = 99.99
N_CAL, N_CLF, N_TEST = 30_000, 50_000, 100_000
POOL_TRAIN_EACH = 100_000
STRENGTHS = [0.0, 1.0, 2.0]
SEED = 42
ARCHS = ["bx_single", "bx_lin10", "bx_cross", "df_single", "df_lin20", "df_cross"]


def find_file(fname):
    for d in [DATA_DIR, os.path.join(DATA_DIR, "python"), "."]:
        cand = os.path.join(d, fname)
        if os.path.exists(cand):
            return cand
    raise FileNotFoundError(f"Cannot find {fname} in {DATA_DIR} or current folder.")


# ------------------------------------------------------------------ source pools (loaded once, paired draws)
Xtr = np.load(find_file("0528_X_train_0_770.npy"), mmap_mode="r")
Xte = np.load(find_file("0528_X_test_0_770.npy"), mmap_mode="r")
ytr = np.load(find_file("0528_y_train_0_770.npy")).astype(int)
yte = np.load(find_file("0528_y_test_0_770.npy")).astype(int)

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(ytr))
cal_pool = np.sort(perm[:POOL_TRAIN_EACH])
clf_pool = np.sort(perm[POOL_TRAIN_EACH:2 * POOL_TRAIN_EACH])


def load_pool(X, idx):
    out = np.empty((len(idx), 2 * T_BINS), np.int16)
    for i in range(0, len(idx), 20_000):
        out[i:i + 20_000] = np.rint(X[idx[i:i + 20_000], :2 * T_BINS]).astype(np.int16)
    return out


print("loading source-shot pools ...")
S_cal, y_cal_pool = load_pool(Xtr, cal_pool), ytr[cal_pool]
S_clf, y_clf_pool = load_pool(Xtr, clf_pool), ytr[clf_pool]
S_te, y_te_pool = load_pool(Xte, np.arange(len(yte))), yte
idx_cal = rng.integers(0, len(y_cal_pool), size=(N_CAL, N_QUBITS))
idx_clf = rng.integers(0, len(y_clf_pool), size=(N_CLF, N_QUBITS))
idx_te = rng.integers(0, len(y_te_pool), size=(N_TEST, N_QUBITS))
Y_cal, Y_clf, Y_te = y_cal_pool[idx_cal], y_clf_pool[idx_clf], y_te_pool[idx_te]

t_sec = np.arange(T_BINS, dtype=np.float64) / FS
carriers = np.exp(1j * 2.0 * np.pi * IF_FREQS[:, None] * t_sec[None, :])
LIN = np.array([[1.00, 0.15, 0.04, 0.01, 0.00], [0.15, 1.00, 0.16, 0.04, 0.01], [0.04, 0.16, 1.00, 0.14, 0.04],
                [0.01, 0.04, 0.14, 1.00, 0.15], [0.00, 0.01, 0.04, 0.15, 1.00]])
PH = np.array([[0.00, 0.28, 0.08, 0.00, 0.00], [-0.26, 0.00, 0.30, 0.08, 0.00], [0.07, -0.28, 0.00, 0.26, 0.07],
               [0.00, 0.07, -0.25, 0.00, 0.28], [0.00, 0.00, 0.07, -0.27, 0.00]])


def synth(S, idx, Y, s, chunk=5000):
    n = len(idx)
    L = np.eye(N_QUBITS) + s * (LIN - np.eye(N_QUBITS))
    P = s * PH
    I_out = np.empty((n, T_BINS), np.int16)
    Q_out = np.empty((n, T_BINS), np.int16)
    for a in range(0, n, chunk):
        b = min(a + chunk, n)
        bb = np.empty((b - a, N_QUBITS, T_BINS), np.complex128)
        for q in range(N_QUBITS):
            rows = S[idx[a:b, q]].astype(np.float64)
            bb[:, q, :] = rows[:, 0::2] + 1j * rows[:, 1::2]
        sgn = 2.0 * Y[a:b] - 1.0
        theta = sgn @ P.T
        prev = np.zeros_like(sgn)
        prev[:, 1:] = sgn[:, :-1]
        bb = bb * ((1.0 + 0.12 * s * prev)[:, :, None] * np.exp(1j * theta[:, :, None]))
        bb = np.einsum("pq,bqt->bpt", L, bb)
        bus = np.einsum("bkt,kt->bt", bb, carriers) / np.sqrt(N_QUBITS)
        I_out[a:b] = np.clip(np.rint(bus.real), -32768, 32767).astype(np.int16)
        Q_out[a:b] = np.clip(np.rint(bus.imag), -32768, 32767).astype(np.int16)
    return I_out, Q_out


# ------------------------------------------------------------------ weight tables
def ema_1d(x, alpha):
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = alpha * x[t] + (1.0 - alpha) * out[t - 1]
    return out


def premix(q, w_mf_I, w_mf_Q, w_rmf_I, w_rmf_Q, W):
    c, sn = np.cos(2.0 * np.pi * IF_FREQS[q] * t_sec), np.sin(2.0 * np.pi * IF_FREQS[q] * t_sec)
    W[q, 0, 0], W[q, 0, 1] = w_mf_I * c, w_mf_I * sn
    W[q, 1, 0], W[q, 1, 1] = -w_mf_Q * sn, w_mf_Q * c
    W[q, 2, 0], W[q, 2, 1] = w_rmf_I * c, w_rmf_I * sn
    W[q, 3, 0], W[q, 3, 1] = -w_rmf_Q * sn, w_rmf_Q * c


def box_tables():
    rect = np.zeros(T_BINS)
    rect[BOX_WIN[0]:BOX_WIN[1]] = 1.0
    W = np.zeros((N_QUBITS, 4, 2, T_BINS))
    z = np.zeros(T_BINS)
    for q in range(N_QUBITS):
        premix(q, rect, rect, z, z, W)
    return W


def df_tables(I_raw, Q_raw, Y):
    S_raw = I_raw.astype(np.float64) + 1j * Q_raw.astype(np.float64)
    W = np.zeros((N_QUBITS, 4, 2, T_BINS))
    for q in range(N_QUBITS):
        Zq = S_raw * np.exp(-1j * 2.0 * np.pi * IF_FREQS[q] * t_sec)[None, :]
        Iq, Qq, yq = Zq.real, Zq.imag, Y[:, q]
        m0, m1 = yq == 0, yq == 1
        mI0, mQ0, vI0, vQ0 = Iq[m0].mean(0), Qq[m0].mean(0), Iq[m0].var(0), Qq[m0].var(0)
        mI1, mQ1, vI1, vQ1 = Iq[m1].mean(0), Qq[m1].mean(0), Iq[m1].var(0), Qq[m1].var(0)
        mtv = np.c_[Iq.mean(1), Qq.mean(1)]
        C0, C1 = mtv[m0].mean(0), mtv[m1].mean(0)
        relax = m1 & (np.linalg.norm(mtv - C0, axis=1) < 0.5 * np.linalg.norm(C0 - C1))
        wmI = ema_1d(mI1 - mI0, EMA_ALPHA) / (0.5 * (vI0 + vI1))
        wmQ = ema_1d(mQ1 - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQ1))
        if relax.sum() >= 20:
            mIR, mQR, vIR, vQR = Iq[relax].mean(0), Qq[relax].mean(0), Iq[relax].var(0), Qq[relax].var(0)
            wrI = ema_1d(mIR - mI0, EMA_ALPHA) / (0.5 * (vI0 + vIR))
            wrQ = ema_1d(mQR - mQ0, EMA_ALPHA) / (0.5 * (vQ0 + vQR))
        else:
            wrI = wrQ = np.zeros(T_BINS)
        premix(q, wmI, wmQ, wrI, wrQ, W)
    return W


def accumulate(I_raw, Q_raw, W, chunk=5000):
    N = len(I_raw)
    out = np.empty((N, N_QUBITS, 4))
    WI = W[:, :, 0, :].reshape(N_QUBITS * 4, T_BINS).T
    WQ = W[:, :, 1, :].reshape(N_QUBITS * 4, T_BINS).T
    for s in range(0, N, chunk):
        e = min(s + chunk, N)
        out[s:e] = (I_raw[s:e].astype(np.float64) @ WI + Q_raw[s:e].astype(np.float64) @ WQ).reshape(e - s, N_QUBITS, 4)
    return out


def normalize(Ac, At):
    mu = Ac.mean(0)
    A = np.percentile(np.abs(Ac - mu), CLIP, axis=0)
    n = np.ceil(np.log2(np.maximum(A, 1e-9)))
    f = lambda a: np.clip((a + 2.0 ** n - mu) / 2.0 ** (n + 1), 0, 1) - 0.5
    return f(Ac), f(At)


# ------------------------------------------------------------------ architectures
def quad_cross(V, q, nf):
    """Self-quadratic terms of qubit q's first nf features + cross products with neighbours' first TWO features."""
    vk = V[:, q, :nf]
    self_q = [vk[:, i] * vk[:, j] for i in range(nf) for j in range(i, nf)]
    cross = []
    for j in range(N_QUBITS):
        if j != q:
            vj = V[:, j, :]
            cross += [vk[:, 0] * vj[:, 0], vk[:, 0] * vj[:, 1], vk[:, 1] * vj[:, 0], vk[:, 1] * vj[:, 1]]
    return np.stack(self_q, 1), np.stack(cross, 1)


def features(arch, Vbx, Vdf, q):
    if arch == "bx_single":
        return Vbx[:, q, :2]
    if arch == "bx_lin10":
        return Vbx[:, :, :2].reshape(len(Vbx), -1)
    if arch == "bx_cross":
        sq, cr = quad_cross(Vbx, q, 2)
        return np.c_[Vbx[:, :, :2].reshape(len(Vbx), -1), sq, cr]
    if arch == "df_single":
        return Vdf[:, q, :]
    if arch == "df_lin20":
        return Vdf.reshape(len(Vdf), -1)
    if arch == "df_cross":
        sq, cr = quad_cross(Vdf, q, 4)
        return np.c_[Vdf.reshape(len(Vdf), -1), sq, cr]
    raise ValueError(arch)


def best_threshold(s, y):
    order = np.argsort(s)
    ss, ys = s[order], y[order]
    n0 = (ys == 0).sum()
    c1 = np.r_[0, np.cumsum(ys == 1)]
    c0 = np.r_[0, np.cumsum(ys == 0)]
    k = int(np.argmin(c1 + (n0 - c0)))
    if k == 0:
        return ss[0] - 1.0
    if k == len(ss):
        return ss[-1]
    return 0.5 * (ss[k - 1] + ss[k])


# hardware cost per qubit: front-end MACs per shot, classifier multiplications (weights incl. bias + products)
nb = BOX_WIN[1] - BOX_WIN[0]
COST = {"bx_single": (4 * nb, 3), "bx_lin10": (4 * nb, 11), "bx_cross": (4 * nb, 30 + 19),
        "df_single": (8 * T_BINS, 5), "df_lin20": (8 * T_BINS, 21), "df_cross": (8 * T_BINS, 47 + 26)}

# ------------------------------------------------------------------ sweep
W_box = box_tables()
for s in STRENGTHS:
    print(f"\n=== crosstalk strength s = {s} ({'CONTROL: no crosstalk' if s == 0 else 'x original model'}) ===")
    I_cal, Q_cal = synth(S_cal, idx_cal, Y_cal, s)
    I_clf, Q_clf = synth(S_clf, idx_clf, Y_clf, s)
    I_te, Q_te = synth(S_te, idx_te, Y_te, s)
    W_df = df_tables(I_cal, Q_cal, Y_cal)
    Vdf_c, Vdf_t = normalize(accumulate(I_clf, Q_clf, W_df), accumulate(I_te, Q_te, W_df))
    Vbx_c, Vbx_t = normalize(accumulate(I_clf, Q_clf, W_box), accumulate(I_te, Q_te, W_box))

    corr, fids = {}, {}
    for arch in ARCHS:
        corr[arch] = np.zeros(Y_te.shape, bool)
        fids[arch] = []
        for q in range(N_QUBITS):
            Fc = np.c_[np.ones(len(Vdf_c)), features(arch, Vbx_c, Vdf_c, q)]
            Ft = np.c_[np.ones(len(Vdf_t)), features(arch, Vbx_t, Vdf_t, q)]
            w = np.linalg.solve(Fc.T @ Fc + RIDGE_LAMBDA_REL * len(Fc) * np.eye(Fc.shape[1]),
                                Fc.T @ (2.0 * Y_clf[:, q] - 1.0))
            pq = (Ft @ w) > best_threshold(Fc @ w, Y_clf[:, q])
            corr[arch][:, q] = pq == (Y_te[:, q] == 1)
            fids[arch].append(0.5 * (np.mean(~pq[Y_te[:, q] == 0]) + np.mean(pq[Y_te[:, q] == 1])))

    base = corr["bx_single"].mean(1)
    print(f"{'architecture':<12s} {'mean F':>7s} {'gain vs bx_single (pts)':>25s} {'front MACs/q':>13s} {'clf mults/q':>12s}")
    for arch in ARCHS:
        d = corr[arch].mean(1) - base
        print(f"{arch:<12s} {np.mean(fids[arch]):7.4f} {100 * d.mean():+10.2f} +/- {100 * d.std() / np.sqrt(len(d)):.2f}"
              f" {COST[arch][0]:13d} {COST[arch][1]:12d}")
    d = corr["df_cross"].mean(1) - corr["bx_cross"].mean(1)
    print(f"df_cross minus bx_cross: {100 * d.mean():+.2f} +/- {100 * d.std() / np.sqrt(len(d)):.2f} points")

print("\nHow to read: at s=0 all gains should be ~0 (control). If bx_cross ~ df_cross at s>0, the crosstalk")
print("benefit comes from the cross-terms, not from the WMF/RMF front end, and plain boxcar is enough.")
print("SYNTHETIC data: report as a pipeline/crosstalk-model benchmark only.")

loading source-shot pools ...

=== crosstalk strength s = 0.0 (CONTROL: no crosstalk) ===
architecture  mean F   gain vs bx_single (pts)  front MACs/q  clf mults/q
bx_single     0.8763      +0.00 +/- 0.00          1800            3
bx_lin10      0.8763      +0.00 +/- 0.01          1800           11
bx_cross      0.8763      -0.01 +/- 0.01          1800           49
df_single     0.8819      +0.55 +/- 0.03          4000            5
df_lin20      0.8819      +0.55 +/- 0.03          4000           21
df_cross      0.8817      +0.54 +/- 0.03          4000           73
df_cross minus bx_cross: +0.54 +/- 0.03 points

=== crosstalk strength s = 1.0 (x original model) ===
architecture  mean F   gain vs bx_single (pts)  front MACs/q  clf mults/q
bx_single     0.8448      +0.00 +/- 0.00          1800            3
bx_lin10      0.8584      +1.35 +/- 0.03          1800           11
bx_cross      0.8624      +1.76 +/- 0.03          1800           49
df_single     0.8574      +1.26 +/- 0.04        